In [ ]:
!pip install -q langchain langchain-community langchain-core langchain-groq chromadb sentence-transformers pypdf python-dotenv

In [ ]:
import os

os.environ["GROQ_API_KEY"] = "<CLAVE_REVOCADA>"

DATA_DIR = "data"
CHROMA_DIR = "chroma_db"
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(CHROMA_DIR, exist_ok=True)

print("¡Entorno configurado correctamente!")

¡Entorno configurado correctamente!


In [ ]:
doc_path = os.path.join(DATA_DIR, "documento_conocimiento.txt")

contenido_base = """
Base de Conocimientos Oficial - Soporte TI Duoc UC
--------------------------------------------------
1. Bloqueo de cuentas: Si un usuario olvida su contraseña o la cuenta se bloquea tras 3 intentos fallidos, debe contactar al administrador de red local o utilizar el portal de autoservicio para restablecerla mediante correo institucional.
2. Configuración de VPN: Para acceder remotamente a los laboratorios, es obligatorio utilizar la VPN corporativa con autenticación de doble factor (2FA).
3. Impresoras de red: Si una impresora muestra el estado 'En pausa', reinicie el servicio de cola de impresión en Windows (services.msc -> Cola de impresión -> Reiniciar).
4. Actualizaciones de software: Las actualizaciones críticas del sistema operativo se despliegan automáticamente todos los viernes a las 22:00 hrs.
"""

with open(doc_path, "w", encoding="utf-8") as f:
    f.write(contenido_base)

print(f"Documento de conocimiento creado en: {doc_path}")

Documento de conocimiento creado en: data/documento_conocimiento.txt


In [ ]:
from langchain_core.prompts import ChatPromptTemplate

SYSTEM_PROMPT = """Eres un asistente de soporte de TI profesional, amable y conciso.
Tu objetivo es responder únicamente utilizando la información proporcionada en el contexto.

Instrucciones estrictas:
1. Responde ÚNICAMENTE basándote en la información presente en el contexto.
2. Si la respuesta NO está explícitamente en el contexto, debes responder exactamente:
   "Lamentablemente no poseo esa información en mi base de conocimientos actual."
3. NO inventes, NO supongas y NO uses conocimientos previos externos al contexto.
4. Mantén un tono respetuoso y profesional.
5. Usa viñetas o formato estructurado si la respuesta incluye múltiples pasos o acciones.

Contexto:
{context}

Pregunta del usuario:
{question}
"""

PROMPT_SISTEMA = ChatPromptTemplate.from_template(SYSTEM_PROMPT)
print("Prompt de sistema cargado con éxito.")

Prompt de sistema cargado con éxito.


In [ ]:
from langchain_community.document_loaders import TextLoader, DirectoryLoader, PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_groq import ChatGroq
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

def construir_rag():
    loaders = [
        DirectoryLoader(DATA_DIR, glob="**/*.txt", loader_cls=TextLoader, loader_kwargs={"encoding": "utf-8"}),
        DirectoryLoader(DATA_DIR, glob="**/*.pdf", loader_cls=PyPDFLoader)
    ]

    documents = []
    for loader in loaders:
        try:
            documents.extend(loader.load())
        except Exception:
            pass

    text_splitter = RecursiveCharacterTextSplitter(chunk_size=600, chunk_overlap=80)
    chunks = text_splitter.split_documents(documents)

    # 3. Embeddings y Base de Datos Vectorial
    embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
    vectorstore = Chroma.from_documents(documents=chunks, embedding=embeddings, persist_directory=CHROMA_DIR)

    retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

    # 4. Configurar Modelo LLM en Groq
    llm = ChatGroq(
        model="qwen/qwen3.8-27b",
        temperature=0.1,
        max_tokens=500,
        api_key="<CLAVE_REVOCADA>"
    )

    # 5. Función auxiliar para formatear los documentos recuperados
    def format_docs(docs):
        return "\n\n".join(doc.page_content for doc in docs)

    rag_chain = (
        {"context": retriever | format_docs, "question": RunnablePassthrough()}
        | PROMPT_SISTEMA
        | llm
        | StrOutputParser()
    )

    return rag_chain, retriever

# Inicializar el agente
agente, retriever = construir_rag()
print("¡Pipeline RAG construido y listo para usar!")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

¡Pipeline RAG construido y listo para usar!


In [ ]:
pregunta = "¿Qué debo hacer si se bloquea mi cuenta?"

print(f"--- CONSULTA ---\n{pregunta}\n")

# Obtener respuesta del agente
respuesta = agente.invoke(pregunta)
print(f"--- RESPUESTA DEL AGENTE ---\n{respuesta}\n")

docs_usados = retriever.invoke(pregunta)
print("--- FUENTES CONSULTADAS ---")
for i, doc in enumerate(docs_usados, 1):
    print(f"\n[Fuente {i}]: {doc.page_content}")

--- CONSULTA ---
¿Qué debo hacer si se bloquea mi cuenta?

--- RESPUESTA DEL AGENTE ---
Si tu cuenta se bloquea tras 3 intentos fallidos, debes realizar una de las siguientes acciones:

*   Contactar al administrador de red local.
*   Utilizar el portal de autoservicio para restablecerla mediante correo institucional.

--- FUENTES CONSULTADAS ---

[Fuente 1]: Base de Conocimientos Oficial - Soporte TI Duoc UC
--------------------------------------------------
1. Bloqueo de cuentas: Si un usuario olvida su contraseña o la cuenta se bloquea tras 3 intentos fallidos, debe contactar al administrador de red local o utilizar el portal de autoservicio para restablecerla mediante correo institucional.
2. Configuración de VPN: Para acceder remotamente a los laboratorios, es obligatorio utilizar la VPN corporativa con autenticación de doble factor (2FA).

[Fuente 2]: Base de Conocimientos Oficial - Soporte TI Duoc UC
--------------------------------------------------
1. Bloqueo de cuentas: Si un

In [ ]:
!pip install -q gradio

import gradio as gr

def responder_chatbot(mensaje, historial):
    try:
        # Invoca al agente RAG con la pregunta del usuario
        respuesta = agente.invoke(mensaje)

        docs_usados = retriever.invoke(mensaje)
        fuentes_str = "\n\n**Fuentes consultadas:**\n"
        for i, doc in enumerate(docs_usados, 1):
            fuentes_str += f"- *{doc.page_content[:100]}...*\n"

        return respuesta + fuentes_str
    except Exception as e:
        return f"Ocurrió un error al procesar la consulta: {str(e)}"

demo = gr.ChatInterface(
    fn=responder_chatbot,
    title="🤖 Asistente RAG - Soporte TI ",
    description="Haz tus consultas basadas en la base de conocimientos interna. El agente responderá estrictamente usando los documentos cargados.",
    textbox=gr.Textbox(placeholder="Escribe tu pregunta aquí (ej. ¿Qué hacer si se bloquea mi cuenta?)...", container=False, scale=7)
)

demo.launch(share=True, debug=False)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://37dae18a657b75a66b.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
